In [1]:
import pandas as pd
import numpy as np
from datetime import datetime
import sys
import warnings
import nest_asyncio
nest_asyncio.apply()  # 중첩 이벤트 루프 허용
import os
import json
import logging
import asyncio
import pandas as pd
import openai
from datetime import datetime
import sys
from typing import List, Dict, Optional
from tenacity import retry, stop_after_attempt, wait_exponential
import re

warnings.simplefilter(action='ignore', category=FutureWarning) # FutureWarning 제거

In [2]:
df = pd.read_excel('../../data/centum_data/21.11-24.6환자 CC_PI_치료계획.xlsx')
df['날짜'] = pd.to_datetime(df['날짜'])
df = df.iloc[:,1:]

df.columns = df.columns.str.strip()
df = df[['환자번호', '날짜', 
        # 'CC', '약', '장치', '습관', '찜질', '마사지, 스트레칭', 'PI',  # 처리 됨.
        'CMO','MMO', 'Cap.pal', 'M.pal', 'Noise', 'Loading', 'Occlusion', 'OJ/OB',
        'Class', 'Midline Shift', 'Deviation', 'CR-CO', 'Tongue ridging',
        'Mucosal ridging', 'Ultrasono', 'Rt', 'Lt',
        'Lateral excursion Protrusive excursion', 'End feel', 
    #    '치료계획', # 필요 없을 듯.
    #    'T-scan 악화/개선', 'CBCT 악화/개선', 'CBCT 판독소견' # 고유겂 NaN    
    ]]

df = df[['환자번호', '날짜',
        'CMO','MMO', 'Cap.pal', 'M.pal', 'Noise', 'Loading', 'Occlusion', 'OJ/OB',
        'Class', 'Midline Shift', 'Deviation', 'CR-CO', 'Tongue ridging',
        'Mucosal ridging', 'Ultrasono', 'Rt', 'Lt',
        'Lateral excursion Protrusive excursion', 'End feel']]

In [3]:
df.columns

Index(['환자번호', '날짜', 'CMO', 'MMO', 'Cap.pal', 'M.pal', 'Noise', 'Loading',
       'Occlusion', 'OJ/OB', 'Class', 'Midline Shift', 'Deviation', 'CR-CO',
       'Tongue ridging', 'Mucosal ridging', 'Ultrasono', 'Rt', 'Lt',
       'Lateral excursion Protrusive excursion', 'End feel'],
      dtype='object')

### CMO & MMO
CMO : 입을 편하게 벌릴 수 있는 최대 크기 \
MMO : 입을 아프더라도 벌릴 수 있는 최대 크기
- 스킴
    - 벌려지는 정도 & 위치 & 처방 종류
    - 화살표 이후의 정보 삭제
    - 고착, 스프레이 필요 없음


In [4]:
df.sample(10).MMO


168              50mm BOTH M--> mm after spray and stretch
17145    38mm Lt) cap, 광대부착부 M--> 43mm after spray and ...
7580           45mm no pain --> mm after spray and stretch
25477                33mm --> 40mm after spray and stretch
1941           48mm NO PAIN --> mm after spray and stretch
22406         50mm (no pain)--> mm after spray and stretch
6232         45mm (no pain) --> mm after spray and stretch
5406            47mm no pain--> mm after spray and stretch
9186         50mm (NO PAIN) --> mm after spray and stretch
10651          52mm (no pain)-> mm after spray and stretch
Name: MMO, dtype: object

In [5]:
df.sample(10).CMO

11713    44mm --> mm after spray and stretch
1642     60mm --> mm after spray and stretch
3761     31mm --> mm after spray and stretch
5535     40mm --> mm after spray and stretch
12544    43mm --> mm after spray and stretch
6236     30mm --> mm after spray and stretch
13553    52mm --> mm after spray and stretch
18367                                  40 mm
24246    33mm --> mm after spray and stretch
6354                                     NaN
Name: CMO, dtype: object

### Deviation
입을 벌리면서 위 턱에 대해 아래 턱이 어떻게 움직이는지 경로를 나타내는 열

- 스킴
    - 방향 & 이동 경로 & 치우침의 강도 or 증상
    - L이 가장 안좋음. S는 낫배드.
    - L, S가 중요하며, 강도는 크게 중요하지 않음.
    고착 후 등 다 필요 없다.

In [6]:
df.Deviation.unique()

array([nan, 'L (Lt)', '-', 'R', 'S', '왼L', '왼 L', 'RT) L', '오른쪽 S',
       '오른쪽 S, 벌리고 다물때', 'Rt)L', 'Lt) L', 'RT', '오 L', '오 S', 'Rt', '오L',
       'Lt', 'Lt L 심함', '오', '오른쪽', 'LT', 'L', 'Lt) S', 'Rt) L',
       '안풀고도 심함,풀어서 L 심함', 'S로 약하게', 'S (Rt)', 'Lt 많이 좋아짐', '오른쪽L',
       '오른쪽L /고착 후  오L없어지고 S로 바뀜', 'Lt)L', 'RT)L 약간', 'RT)L',
       'mild L deviation', 's', '없음', 'Lt L', '오른쪽 L', 'n/s', 'Rt) S 거칠게',
       'Rt) S ->거의없음', 'Rt)S', 'Lt S', 'Rt L', 'Rt) L 의심', 'Rt) S',
       'Lt L -> S after spray', 'L왼', '왼L심함', '오s', 'S 심함', 'L,S 중간정도',
       'S(Rt)', 's 심함', '약간 Rt', '오른쪽 L 약간', 'Rt) L 마지막에 원래대로 돌아옴',
       'LT) L', 'Lt)L 있었던적 있음', 'Lt) S,L 중간', 'S (왼쪽으로 갔다가 가운데)', '오l',
       's심함', 'RT L', '왼L++', '안틀면 오L', '왼쪽L', '왼쪽l', '왼쪽 L 거의없음', 'L 살짝',
       'Z', 'both L', '약간 s', 'R 많이 기울어짐', '오S', 'S심함', '왼S',
       'L->S(고착 후)', 'S 약하게', '왼L (양쪽 걸린 느낌)', '양쪽L',
       'S 벌릴때 오른쪽->다물때 왼쪽->', '오L+S중간', '오l경향성', '살짝 L', 'S, L 중간',
       'S (가끔씩 Rt L)', 'R(S)', 'R(L)', 'L(L)', 'R

### Cap.pal & M.pal
씹을때의, 근육의 통증? Cap = 뼈, M 근육

- 스킴
    - 위치 & 강도 & 조건부 상황
    - +,- > + 순으롤 아파짐. (공백은 + 하나)
    - -가 되는 것이 목표



In [7]:
df[(df['Cap.pal'].notna()) & (df['Cap.pal'] != '-')].sample(20)['Cap.pal']

11048           lt cap
6485              lt)+
13972         Lt) cap+
22393     Rt capsule불편
27413            Rt) +
13445         Rt) cap+
20544          Lt cap+
27989            Lt) +
17821       Rt + 약간 감소
26674            lt) +
8137              Rt)+
3268             rt )+
19561        Rt cap 괜찮
3908           Lt) cap
13086       Rt) cap+/-
8430           Lt (뒤쪽)
26371    Both)+(Rt>Lt)
1567              Lt)+
27788            Lt) +
23512             Rt +
Name: Cap.pal, dtype: object

In [8]:
df[(df['M.pal'].notna()) & (df['Cap.pal'] != '-')].sample(20)['Cap.pal']

18733    +/- (Lt cap pain)
25228                  NaN
805          -/캡술 아래 아픔 감소
27452                rt) +
18663                  n/s
23130             Rt +(살짝)
25505             Lt) cap+
22581                  Rt+
3274                rt)+/-
6690                  Rt)+
708                    NaN
6529        both) +(rt가 더)
12936          Lt)+ 턱떨림 심함
16425        both) + Rt>Lt
19530           Lt capsule
3909               Lt) cap
26735           Rt cap)+/-
13921              RT)CAP+
168               BOTH CAP
25226                  NaN
Name: Cap.pal, dtype: object

### Noise
관절 움직임 시 소음?

- 스킴
    - 위치 & 소음종류(click, snap, pop, etc) & (조건부 상황 or 빈도)
    - popping : 밖에서 들리는 소리, click : 밖에서 안들리는 소리, creptitus : 뼈갈리는 소리
    - creptitus, popping : bad. click : good. bad 에서 good 으로 가는게 목표
    



In [9]:
df[(df['Noise'].notna()) & (df['Noise'] != '-')].sample(20)['Noise']

14931                       양쪽 Click (왼쪽이 더함)
22991                              both click
1670                 both)click (LT>RT) 거의 없음
15882                          Lt) popping 가끔
2313                                 Rt click
14712                                  Rt) 소리
12376                                RT click
18966                            Rt +click 작게
15803                                Lt)click
27695        both closing click , Lt openning
11687                 Noise  RT click 벌리고 닫을때
8990     Rt) click 살짝/ Rt로 벌렸을때 Rt)popping 1번
10338                  Lt closing click 남아 있음
3331                        초음파 both)click 의심
4346                                Lt) click
8680                                rt) click
3737                            마지막 Rt) click
6827                                Lt) click
25653                           Both)click 살짝
9887                              오늘은 click X
Name: Noise, dtype: object

### Occlusion
Occlusal ? 교합 혹은 다물었을때 증상? 
, 교합

- 스킴
    - 위치 & 상악 하악의 닿는 이빨 배열? & 특징
- 예시
    - clear
        - none
    - issue
        



In [10]:
df[(df['Occlusion'].notna()) & (df['Occlusion'] != '-')].sample(20)['Occlusion']

13601               4567/4567
10012    오른쪽 5번 안 닿음 -> 4번 발치
18563                 양쪽 4567
20320               4567/4567
12276               4567/4567
6109                4567/4567
26483                 567/567
14483               4567/4567
3382                4567/4567
3025                4567/4567
19088               Both 4567
9465                  BOTH 67
15755               4567/4567
1003                4567/4657
16922                 567/567
823                 4567/4567
8585                  567/567
10382               양쪽 7번만 닿음
6859                4567/4567
24426               4567/4567
Name: Occlusion, dtype: object

### OJ/OB
Overbite는 상악(윗니)의 앞니가 하악(아랫니)의 앞니를 덮는 정도를 나타냄 (정상적인 오버바이트는 약 2~4mm 정도가 적당)  \
Overjet는 윗니가 아랫니보다 얼마나 튀어나와 있는지를 측정하는 것 (정상적인 오버젯은 보통 2~3mm)

OB = 2mm 보다 작으면 안좋음. 마이너스가 안좋은거 
OJ = 3mm 이상 커지만 안좋음.

- 스킴
    - 위치 & 강도 & 조건부 상황
- 예시
    - clear
        - none
    - issue
        - 3 / 0 - open bite가능성 
        - 4/3 
        - 3/4
        - 0.5mm/0mm




In [11]:
df[(df['OJ/OB'].notna()) & (df['OJ/OB'] != '-')].sample(20)['OJ/OB']

17176           2/2.5
3779        2mm/1.5mm
27664             4/2
24600           1/0.5
2603              3/2
3370              3/0
2387           2/-0.5
23127             2/2
26419             2/2
22096           3/0.5
16074             4/2
331             1/0.5
17945             n/s
9159              2/1
25995       #11기준 0/0
10735    #11 0.5/0.58
12247             2/1
2994              2/2
25810             3/2
23710             3/2
Name: OJ/OB, dtype: object

### Class
환자 분류>?

- 스킴
    - 위치 & 강도 & 조건부 상황
- 예시
    - clear
        - none
    - issue
        - s?




In [12]:
df[(df['Class'].notna()) & (df['Class'] != '-')].sample(20)['Class']

15070    3
25818    2
8167     3
24133    1
17580    1
15269    2
6681     3
21361    3
13828    2
6450     1
11456    s
6338     1
8607     3
1050     1
7361     3
7881     3
12444    1
10085    s
27402    2
16788    1
Name: Class, dtype: object

### Midline Shift
이빨 중앙 라인에서 왜도

- 스킴
    - 경향성 & 거리
- 예시
    - clear
        - none
    - issue
        - n/s
        - 하악 왼쪽 2mm




In [13]:
df[(df['Midline Shift'].notna()) & (df['Midline Shift'] != '-')].sample(20)['Midline Shift']

24272           하 왼 1
1022       상악 오른쪽 3mm
15021           하 왼 2
8409              상왼2
26328             하왼2
4199          하 왼 1mm
7607            상악 왼3
2449              상왼5
18484     상악 왼쪽으로 3mm
1286          하 오 2mm
3939              하왼2
3084       하악 오른쪽 2mm
26964           하 왼 1
6588           상악왼쪽 2
22581     하악오른쪽으로 1mm
3646        하 왼 1.5mm
7595          상악 왼쪽 3
26034    상악이 왼쪽으로 4mm
13728    하악이 왼쪽으로 1mm
27444           하왼0.5
Name: Midline Shift, dtype: object

### CR-CO
- Centric Relation
    - Centric Relation은 교합이 가장 안정적이고 균형 잡힌 상태일 때, 즉 두 턱이 제대로 맞물리는 위치를 나타냅니다.
    - CR은 근육과 인대가 최대로 긴장되거나 최적의 위치에 있을 때로, 이 상태에서 하악을 상악과 맞추는 것이 중요합니다.
- Centric Occlusion
    - 실제로 두 턱이 닫힐 때, 즉 치아가 맞물리는 상태를 말합니다. 하악의 치아가 상악의 치아와 접촉하는 지점으로, Centric Occlusion은 교합의 "물어보는" 상태를 의미합니다.
    - CO는 일반적으로 CR과 일치하는 것이 이상적이나, 때로는 CR과 CO가 일치하지 않는 경우도 있을 수 있습니다. 이런 경우에는 교정치료가 필요할 수 있습니다.

어느쪽으로든 2mm 이상이면 안좋다. 두 케이스가 이정도 차이 이상이면 병적으로 의심됨. 방향 상관 없이 다 안좋은거


- 스킴
    - 위치 & 거리
- 예시
    - clear
        - none
    - issue
        - 오른쪽 뒤 0.5-1mm



In [14]:
df[(df['CR-CO'].notna()) & (df['CR-CO'] != '-')].sample(20)['CR-CO']

16927                           매우 큼
24714                     오른쪽 뒤로 2mm
11873                             없음
23853    12번 반대교합 때문에 cr-co 있는걸로 보이심
4618                            거의없음
18873                             없음
21288                        후방 2 mm
5518                         오 뒤 1mm
3510                              없음
20483                          0.5-1
14199                        없으신것 같음
20572                          0.5-1
12496                       후방 1~2mm
12189                             없음
11220                          0.5mm
16344                     왼쪽 뒤로 -2mm
3502                              없음
11988                            뒤 2
15931                       뒤로 1-2mm
10667                             없음
Name: CR-CO, dtype: object

### Tongue ridging
- 혀의 표면에 나타나는 주름이나 능선을 의미합니다. 이는 보통 혀의 중앙에 세로로 나타나는 주름을 가리키며, 혀의 모양에 영향을 미칠 수 있습니다.

마이너스가 제일 좋다. 

- 스킴
    - 강도
- 예시
    - 강한 양성 (++)	혀의 융기가 매우 뚜렷하고 심각한 수준으로 나타나는 경우
    - 약한 양성 (+)	혀의 융기가 경미하게 관찰되는 상태
    - 음성 (-)	혀의 융기가 전혀 없거나 관찰되지 않는 상태
    - 미확인 (n/s)	데이터 확인이 불가능하거나 판단할 수 없는 상태



In [15]:
df[(df['Tongue ridging'].notna()) & (df['Tongue ridging'] != '-')].sample(20)['Tongue ridging']

8914      ++
20811      +
17370      +
3695       +
27847      +
2841       +
2952       +
1912       +
17679      +
11498      +
22142      +
25380      +
17076      +
669        +
15987      +
17893      +
22081    +++
10534      +
8765       +
8838      심함
Name: Tongue ridging, dtype: object

### Mucosal ridging
- 구강 점막(즉, 입안의 내부 표면)에 나타나는 주름이나 능선을 의미합니다. 이는 일반적으로 점막이 늘어나거나 두꺼워지면서 생기는 구조적 변화로, 구강 내 다양한 부위에서 발생할 수 있습니다.

마이너스 가 좋다.

- 스킴
    - 강도
- 예시
    - 강한 양성 (++)	혀의 융기가 매우 뚜렷하고 심각한 수준으로 나타나는 경우
    - 약한 양성 (+)	혀의 융기가 경미하게 관찰되는 상태
    - 음성 (-)	혀의 융기가 전혀 없거나 관찰되지 않는 상태
    - 미확인 (n/s)	데이터 확인이 불가능하거나 판단할 수 없는 상태



In [16]:
df[(df['Mucosal ridging'].notna()) & (df['Mucosal ridging'] != '-')].sample(20)['Mucosal ridging']

24265      +
24396     ++
21661      +
14454      +
7928       +
11352      +
19467      +
20679    +심함
2170      심함
5974       +
53         +
20906     심함
2209      ++
16311      +
22774     ++
18679     ++
20670    +심함
17515      +
1422       +
15083      +
Name: Mucosal ridging, dtype: object

### Rt Lt
- 근육 두께
- 스킴
    - 힘 안줬을 떄 초음파로 근육두께 대고, 
    - 화살표 전후로 힘 안 줬을 때, 꽉 물었을 때.
    - 남자는 1.5, 여자는 1.3 미만으로 가는게 목표.
    - 중요한 스키마

In [17]:
df[(df['Rt'].notna()) & (df['Rt'] != '-')].sample(20)['Rt']

15887                1.16 -> 1.43
16518                1.19 -> 1.51
12231                 0.79 -> 1.2
22777                 1.09 ->1.21
26462                1.21 -> 1.53
9881         0.98 -> 1.3 / ->1.20
1387         1.38 -> 1.66 /->1.62
17977                1.09 -> 1.23
21113                 1.11-> 1.44
28064                  0.83->1.20
765       1.27 ->1.73 /1.09->1.42
4252                 0.88 -> 0.94
11451                0.99 -> 1.41
20063                1.13 -> 1.57
17364                1.24 -> 1.62
17307    1.61 -> 2.24 /1.36->1.87
6587                   1.20->1.48
6617                  1.25-> 1.65
2648                 0.96 -> 1.18
23077                 1.19 -> 1.6
Name: Rt, dtype: object

In [18]:
df[(df['Lt'].notna()) & (df['Lt'] != '-')].sample(20)['Lt']

17414                                         1.02 -> 1.47
382                                           0.96 -> 1.26
25719                                                   ->
5037                                          1.05 -> 1.37
22981                                          1.04-> 1.28
24163                                          1.37 ->1.88
25545                                         1.11 -> 1.51
21104    1.04 -> 1.57 /가운데부위 1.2->1.6/ 1.05->1.42/ 1.15...
9483                                            1.2 ->1.62
19955                                                   ->
19883                                                   ->
9215                                    1.07 -> 1.33/ 1.39
7700                                          1.04 -> 1.19
8319                                 1.11->1.53/0.91->1.15
1578                                           0.86 -> 1.4
22143                                          1.25 ->1.66
27281                                           0.76->1.

### Lateral excursion Protrusive excursion --> 삭제
- Lateral excursion은 하악(아랫니)이 좌우로 움직이는 동작을 말합니다. 즉, 아래턱을 좌우로 한쪽으로 밀었을 때의 움직임을 의미합니다. 이는 턱이 한쪽으로 기울어지는 방향으로 이동하며, 주로 측면 교합을 분석하는 데 사용됩니다.
- Protrusive excursion은 하악이 앞쪽으로 이동하는 동작을 말합니다. 즉, 아래턱을 앞으로 내밀었을 때의 움직임을 의미합니다. 이를 통해 앞쪽 교합을 분석하거나 평가할 수 있습니다.


- 스킴
    - 강도
- 예시
    - issue
        - n/s : 무증상 Ok
        - Lt) PAIN on protrusion, Rt : 왼쪽 특정 위치에 통증, 오른쪽도?



In [19]:
df[(df['Lateral excursion Protrusive excursion'].notna()) & (df['Lateral excursion Protrusive excursion'] != '-')].sample(20)['Lateral excursion Protrusive excursion']

17774                    n/s
17785                    n/s
19473                    n/s
21762                    n/s
19093                    n/s
19632            전체적 limited
17931                    n/s
18908                    n/s
17524                    n/s
17703                    n/s
15386    Rt pain on pro & Rt
18202                    n/s
18204                    n/s
18664                    n/s
18029                    n/s
19635            전체적 limited
20335                    n/s
18732                    n/s
21924                    n/s
21919                    n/s
Name: Lateral excursion Protrusive excursion, dtype: object

### 치료 계획


- 스킴
    - 치료 종류, 장치 유무, 다음 내방 
    - ck = 체크
    - 몇달 후에 보냐가 중요함. 짧은 시일 내 본다는 것은 잘 안낫고있다는 뜻.



In [20]:
df[(df['치료계획'].notna()) & (df['치료계획'] != '-')].sample(20)['치료계획']

KeyError: '치료계획'

In [72]:
df.columns

Index(['환자번호', '날짜', 'CC', '약', '장치 ', '습관', '찜질 ', '마사지, 스트레칭', 'PI', 'CMO',
       'MMO', 'Cap.pal', 'M.pal', 'Noise', 'Loading', 'Occlusion', 'OJ/OB',
       'Class', 'Midline Shift', 'Deviation', 'CR-CO', 'Tongue ridging',
       'Mucosal ridging', 'Ultrasono', 'Rt', 'Lt',
       'Lateral excursion Protrusive excursion', 'End feel', '치료계획',
       'T-scan 악화/개선', 'CBCT 악화/개선', 'CBCT 판독소견'],
      dtype='object')

In [121]:
df[(df['Loading'].notna()) & (df['Loading'] != '-')].sample(20)['Loading']

18380                    n/s
19102              Loading -
25115    LT +click, crepitus
10106                    - -
17785                    n/s
18207                    n/s
19100              Loading -
18115                    n/s
18464                    n/s
18979                    n/s
25121         LT) click 아주약간
17775                    n/s
8144                     교정중
18428                    N/S
18935                    n/s
18664                    n/s
18882                    N/S
18808                    n/s
18523                    n/s
25116    LT +click, crepitus
Name: Loading, dtype: object